In [1]:
# %% [markdown]
# Notebook ETL – leitura em batch, validações e write-back


In [2]:
# %% [code]
import os
import sys
import logging
from rich.console import Console
from rich.logging import RichHandler
from dotenv import load_dotenv

# ── Carrega variáveis de ambiente de .env (opcional) ───────────────────────
load_dotenv()

# ── Ajusta PYTHONPATH para o diretório do projeto ─────────────────────────
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# ── Configuração de logs no notebook ──────────────────────────────────────
console = Console(width=120)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s › %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        RichHandler(
            console=console,
            rich_tracebacks=True,
            show_time=True,
            show_level=True,
            show_path=False,
            markup=True,
        )
    ],
)
log = logging.getLogger(__name__)


In [3]:
# %% [code]
import math
import numpy as np
from typing import Any

def _to_json_safe(x: Any) -> Any:
    """
    Internal helper to convert arbitrary objects into JSON-safe primitives.
    """
    if x is None:
        return None
    if isinstance(x, (int, str, bool)):
        return x
    if isinstance(x, float):
        return None if math.isnan(x) or math.isinf(x) else x
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return None if (math.isnan(x) or math.isinf(x)) else float(x)
    if isinstance(x, (list, tuple, set)):
        return [_to_json_safe(item) for item in x]
    if isinstance(x, dict):
        return {k: _to_json_safe(v) for k, v in x.items()}
    # Fallback: stringify anything else
    return str(x)

def json_safe(obj: Any) -> Any:
    """
    Recursively converts `obj` into structures 100% serializable to JSON.
    """
    return _to_json_safe(obj)


In [4]:
# %% [code]
# Flags de gravação (ajuste conforme necessidade)
WRITE_BACK_ORIGIN = True   # grava na aba-origem (meta*, tiktok*, …)
WRITE_BACK_DEST   = True   # grava nas abas-modelo (modelo*)
DRY_RUN_DEST      = False  # True = simula write-back destino

# Credenciais e identificador da planilha
CREDS_PATH     = os.getenv("GOOGLE_CREDS_PATH", "creds.json")
SPREADSHEET_ID = os.getenv(
    "GOOGLE_SHEET_ID",
    "1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg"
)

# Abas de origem a processar, agrupadas por plataforma
SHEET_NAMES = [
    # Meta (Facebook / Instagram)
    "metaGeral",
    "metaIdade",
    "metaGenero",
    "metaRegiao",
    "metaAlcance",
    # TikTok
    "tiktokGeral",
    "tiktokIdade",
    "tiktokGenero",
    "tiktokRegiao",
    "tiktokAlcance",
    # Pinterest
    "pinterestGeral",
    "pinterestGenero",
    "pinterestIdade",
    "pinterestRegiao",
    "pinterestAlcance",
    # LinkedIn
    "linkedinGeral",
    "linkedinRegiao",
    "linkedinAlcance",
    # Google Analytics
    "GAGeral",
]


In [5]:
#4
# %% [code]
import importlib

# módulos principais e helpers recarregados
import extract.sheets_fetcher               as sf_mod
import treat.treat_pipeline                 as tp_mod
import treat.platforms                      as platforms_mod
import treat.platforms.linkedin             as linkedin_mod
import treat.platforms.tiktok               as tiktok_mod
import treat.platforms.pinterest            as pinterest_mod
import treat.platforms.meta                 as meta_mod
import treat.platforms.ga                   as ga_mod
import load.origin_writer                   as ow_mod
import load.dest_writer                     as dw_mod
import treat.utils.renomeacoes              as rn_mod
import treat.utils.preview_links            as prev_mod
import treat.utils.atribuicoes_via_lookup   as atrib_mod
import treat.utils.substitute_origin_values as sub_mod
import treat.utils.preprocess_utils         as pre_mod
import treat.utils.geo_normalize            as geo_mod

# hot-reload de todos os módulos alterados durante o desenvolvimento
for m in (
    sf_mod,
    tp_mod,
    platforms_mod,
    linkedin_mod,
    tiktok_mod,
    pinterest_mod,
    meta_mod,
    ga_mod,
    ow_mod,
    dw_mod,
    rn_mod,
    prev_mod,
    atrib_mod,
    sub_mod,
    pre_mod,
    geo_mod,
):
    importlib.reload(m)


In [6]:
# %% [code]
# Cell 5: definição do helper run_etl_for_sheet
from typing import Dict
import pandas as pd
import gc
import logging                       # ← adicionado
from pprint import pp

from extract.sheets_fetcher import SheetsFetcher
from treat.treat_pipeline import TreatPipeline
from treat.utils.renomeacoes import renomeacao_geral, renomear_colunas_origem_para_modelo
from treat.utils.campos_calculados import calcular_engajamento_total, gerar_id
from load.origin_writer import write_back_origin
from load.dest_writer import write_back_for_sheet

# Instância única do fetcher (retry/backoff/cache interno)
fetcher = SheetsFetcher(
    spreadsheet_id=SPREADSHEET_ID,
    creds_path=CREDS_PATH,
)

def run_etl_for_sheet(
    *,
    sheet: str,
    wb_origin_flag: bool,
    wb_dest_flag: bool,
    dry_run_dest: bool,
    preloaded_raw: pd.DataFrame,
) -> Dict[str, pd.DataFrame | dict]:
    """
    Executa o fluxo completo para uma aba e devolve:
      { "dest": DataFrame destino (ou vazio), "taxo": relatório de taxonomia }
    """
    # 1) Dados brutos já carregados
    df_raw = preloaded_raw

    # 2) Tratamento via pipeline
    pipeline = TreatPipeline(
        creds_path=CREDS_PATH,
        spreadsheet_id=SPREADSHEET_ID,
        sheet_name=sheet,
        mapping_renomeacao=renomeacao_geral,
        write_back=wb_origin_flag,
    )
    df_ok = pipeline.run(df_raw)

    # 3) Relatório de taxonomia
    taxo_report = getattr(pipeline, "_last_taxo_report", {})
    pp(json_safe(taxo_report), width=120)

    # 4) Write-back na aba de origem (apenas quando não for Pinterest demográfico)
    is_pinterest_dim = sheet.lower() in {
        "pinterestgenero", "pinterestidade", "pinterestregiao"
    }

    if not is_pinterest_dim:
        # grava correções de pré-processamento in-place
        _ = write_back_origin(
            df_raw        = df_raw,
            df_ok         = df_ok,
            creds_path    = CREDS_PATH,
            spreadsheet_id= SPREADSHEET_ID,
            sheet_name    = sheet,
            write_back    = wb_origin_flag,
            dry_run       = not wb_origin_flag,
        )
    else:
        logging.debug(
            "🔸 %s: pulando write-back de origem (ja feito dentro de pipeline)", 
            sheet
        )


    # 5) Preparar DataFrame de destino (modelo)
    df_model = renomear_colunas_origem_para_modelo(df_ok, renomeacao_geral)
    df_model = calcular_engajamento_total(df_model)
    df_model["ID"] = df_model.apply(gerar_id, axis=1)

    # 6) Write-back de destino  ──────────────────────────────────────────────
    if sheet.lower().startswith("ga"):
        logging.info("🔸 %s: write-back de destino ignorado (Google Analytics)", sheet)
        df_dest = pd.DataFrame()      # retorna DataFrame vazio
    else:
        df_dest = write_back_for_sheet(
            df_model,
            sheet_name     = sheet,
            creds_path     = CREDS_PATH,
            spreadsheet_id = SPREADSHEET_ID,
            write_back     = wb_dest_flag,
            dry_run        = dry_run_dest,
        )
        if df_dest is None:
            df_dest = pd.DataFrame()

    # 7) Retorno
    return {"dest": df_dest, "taxo": taxo_report}


In [7]:
# %% [code]
# Cell 6: Processamento em lote das abas
from contextlib import suppress
import gc
import logging
import pandas as pd
from tqdm.auto import tqdm
from load.dest_writer import prefetch_meta

# 1) Leitura batch de todas as abas
all_raw = fetcher.get(SHEET_NAMES)
prefetch_meta(CREDS_PATH, SPREADSHEET_ID)

# 2) Processamento aba a aba
results: dict[str, dict[str, object]] = {}

for sheet in tqdm(SHEET_NAMES, desc="Processando abas"):
    is_ga = sheet.lower().startswith("ga")          # controle extra de log
    if is_ga:
        logging.info("🔸 %s: apenas write-back de origem; destino será ignorado", sheet)

    out = run_etl_for_sheet(
        sheet           = sheet,
        wb_origin_flag  = WRITE_BACK_ORIGIN,
        wb_dest_flag    = WRITE_BACK_DEST,   # flag global — run_etl decide pular GA
        dry_run_dest    = DRY_RUN_DEST,
        preloaded_raw   = all_raw[sheet],
    )

    results[sheet] = {"dest": out["dest"], "taxo": out["taxo"]}

    gc.collect()


09:51:29 INFO     09:51:29 INFO extract.sheets_fetcher › 🔄 batchGet tentativa para ranges: ['metaGeral!A:ZZ',          
                  'metaIdade!A:ZZ', 'metaGenero!A:ZZ', 'metaRegiao!A:ZZ', 'metaAlcance!A:ZZ', 'tiktokGeral!A:ZZ',       
                  'tiktokIdade!A:ZZ', 'tiktokGenero!A:ZZ', 'tiktokRegiao!A:ZZ', 'tiktokAlcance!A:ZZ',                   
                  'pinterestGeral!A:ZZ', 'pinterestGenero!A:ZZ', 'pinterestIdade!A:ZZ', 'pinterestRegiao!A:ZZ',         
                  'pinterestAlcance!A:ZZ', 'linkedinGeral!A:ZZ', 'linkedinRegiao!A:ZZ', 'linkedinAlcance!A:ZZ',         
                  'GAGeral!A:ZZ']

09:51:34 INFO     09:51:34 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaGeral!A1:AQ8776

         INFO     09:51:34 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaIdade!A1:Q4282

         INFO     09:51:34 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaGenero!A1:Q1730

         INFO     09:51:34 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaRegiao!A1:O20004

...

         INFO     09:53:37 INFO extract.sheets_fetcher › 📡 batchGet 1 ranges

09:53:52 INFO     09:53:52 INFO treat.utils.merges.pinterest.pinterest_dimension_pin_id_merge › ✅                      
                  merge_pinterest_dimension – 447291 linhas (gender)

{}


/home/debrito/Documentos/etl_debrito/load/dest_writer.py:180: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  .applymap(_scalar)
/home/debrito/Documentos/etl_debrito/load/dest_writer.py:2: SyntaxWarning: invalid escape sequence '\*'
  """Write‑back otimizado para as abas **modelo\***


APIError: APIError: [400]: Invalid data[0]: This action would increase the number of cells in the workbook above the limit of 10000000 cells.

In [ ]:
#7
# %% [code]
from contextlib import suppress
import gc
import pandas as pd

# ── 1) batchGet único de todas as abas ──────────
all_raw = fetcher.get(SHEET_NAMES)
log.info("[bold green]Abas carregadas:[/] %s", list(all_raw.keys()))

# ── Debug opcional de uma aba específica -----------
with suppress(KeyError):
    dbg = all_raw["linkedinRegiao"]
    # log.debug("linkedinRegiao > colunas=%s\n%s",
    #           dbg.columns.tolist(), dbg.head(2).T)

# ── 2) Processa cada aba em memória ----------------
results: dict[str, dict[str, object]] = {}

for sheet in SHEET_NAMES:
    # log.info("[cyan]▶️  Processando %s …[/]", sheet)

    out = run_etl_for_sheet(
        sheet=sheet,
        wb_origin_flag=WRITE_BACK_ORIGIN,
        wb_dest_flag=WRITE_BACK_DEST,
        dry_run_dest=DRY_RUN_DEST,
        preloaded_raw=all_raw[sheet],
    )

    # guarda apenas destino + relatório de taxonomia
    results[sheet] = {"dest": out["dest"], "taxo": out["taxo"]}

    # loga os shapes resumidos
    shapes = {
        k: f"{v.shape[0]:,}×{v.shape[1]}" if isinstance(v, pd.DataFrame) else "-"
        for k, v in out.items()
    }
    # log.info("[green]Shapes:[/] %s", shapes)

    # libera memória
    gc.collect()


In [ ]:
#8
# %% [code]
from treat.treat_pipeline import BIParamLookup

# — Limpa cache de leituras em lote (SheetsFetcher já instanciado como `fetcher` em células anteriores)
fetcher.refresh(SHEET_NAMES)

# — Limpa cache da parametrização BI em memória
BIParamLookup._df = None
BIParamLookup._last_load = 0.0


In [ ]:
#9
# — Forçar recarregamento dos parâmetros BI em qualquer ponto do notebook —
from treat.bi_param_utils import BIParamLookup

# Zera o cache interno para que a próxima chamada a .df() refaça o carregamento
BIParamLookup._df = None
BIParamLookup._last_load = 0.0


In [ ]:
#10 – Validar consistência de datas entre modelos

from treat.utils.validations import validate_consistent_dates_across_models

# Extrai apenas os DataFrames de destino
dest_dfs = {sheet: info["dest"] for sheet, info in results.items()}

# Roda a validação
df_inconsistencies = validate_consistent_dates_across_models(dest_dfs)

# Exibe resultados
if df_inconsistencies is not None and not df_inconsistencies.empty:
    display(df_inconsistencies)
else:
    print("✅ Nenhuma divergência de start/end entre modelos.")


In [ ]:
import gspread, google.auth
from pprint import pprint

creds, _ = google.auth.load_credentials_from_file(CREDS_PATH, scopes=[
    "https://www.googleapis.com/auth/spreadsheets.readonly"
])
gc = gspread.authorize(creds)
ss = gc.open_by_key(SPREADSHEET_ID)

stats = []
for ws in ss.worksheets():
    rows = ws.row_count
    cols = ws.col_count
    cells = rows * cols
    stats.append((cells, ws.title, rows, cols))

stats.sort(reverse=True)          # maiores primeiro
pprint(stats[:40])                # top 10 abas que mais ocupam células
